# EXP-0028 piezo (bimorfo PVDF, converso y directo) en Colab A100

**Antes de nada**: Entorno de ejecucion > Cambiar tipo > GPU **A100**; active el pago por uso en su cuenta de Colab. Este cuaderno **no pide ni usa ninguna clave** (W&B queda OFFLINE; se sincroniza despues, en local).
Orden: 1) montar Drive, 2) verificar GPU, 3) descomprimir el zip, 4) **prueba de tiempo**, 5) decidir cuantas semillas, 6) lanzar, 7) reanudar si se corta, 8) empaquetar y descargar.
Prioridades: 1 = semillas 4001-4003 de ambos problemas (30 trabajos); 2 = semillas 4004-4005 (20 trabajos).

In [ ]:
# 1. PARAMETROS (edite solo esta celda)
ZIP_EN_DRIVE = "/content/drive/MyDrive/x28_colab/exp0028_colab_bundle.zip"   # el zip que subio a Drive (build_colab_bundle.py)
SHA256_ESPERADO = ""                      # opcional: pegue el SHA256 del zip que imprimio build_colab_bundle.py
OUT = "/content/drive/MyDrive/x28_colab/salida_piezo"          # carpeta PERSISTENTE (Drive): resultados, ledger.jsonl, wandb offline
SCRATCH = "/content/x28_scratch"          # carpeta LOCAL de trabajo (rapida); lo terminado se copia a OUT
WORK = "/content/x28"                     # donde se descomprime el bundle (local)
UNIDADES_POR_HORA = None                  # VERIFICAR en su cuenta de Colab (consumo de la A100 en unidades/hora). No hay valor por defecto: sin el no se proyectan unidades.
PROCESOS_POR_GPU = 1                      # 2 solo si la prueba de concurrencia (celda 5) muestra ganancia de rendimiento
MAX_PRIORIDAD = 1                         # 1 = 3 semillas (4001-4003); suba a 2 despues de ver el costo
TOPE_DESPACHO_H = 11.0                    # horas de esta sesion tras las cuales NO se lanza trabajo nuevo
TOPE_KILL_H = 11.5                        # horas tras las cuales se MATAN los trabajos en curso
MAX_UNIDADES = None                       # opcional: tope de gasto en unidades de computo (exige UNIDADES_POR_HORA)
SOLO_PROBLEMA = ""                       # p. ej. "dfg" para correr solo ese problema del manifiesto (vacio = todos los de la prioridad)
PASOS_PRUEBA = 300                        # pasos de Adam de la prueba de tiempo (200-500)

In [ ]:
# 2. Montar Drive y verificar la GPU
from google.colab import drive
drive.mount("/content/drive")
import subprocess
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
name = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True).stdout
assert ("A100" in name) or ("T4" in name), f"GPU no soportada ({name.strip()}): use T4 o A100 y reinicie."

In [ ]:
# 3. Descomprimir el bundle (sin internet salvo pip) y verificar SHA256
import hashlib, os, shutil, zipfile, subprocess, sys, json
from pathlib import Path
z = Path(ZIP_EN_DRIVE); assert z.exists(), f"no existe {z}: suba el zip a Drive"
h = hashlib.sha256(z.read_bytes()).hexdigest(); print("SHA256 del zip:", h)
assert (not SHA256_ESPERADO) or h == SHA256_ESPERADO.strip().lower(), "el SHA256 del zip NO coincide"
shutil.rmtree(WORK, ignore_errors=True); zipfile.ZipFile(z).extractall(WORK)
bad = []
for line in Path(WORK, "SHA256SUMS.txt").read_text().splitlines():
    digest, name = line.split("  ", 1)
    if hashlib.sha256(Path(WORK, name).read_bytes()).hexdigest() != digest: bad.append(name)
assert not bad, f"archivos alterados: {bad}"
print("bundle verificado:", len(Path(WORK, "SHA256SUMS.txt").read_text().splitlines()), "archivos")
# dependencias: torch y numpy ya vienen en Colab; wandb solo si falta (pip es lo unico que usa internet). NO se hace wandb.login.
try:
    import wandb
except Exception:
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "wandb"], check=False)
RUNNER = f"{WORK}/colab_runner.py"; MANIFEST = f"{WORK}/manifest_piezo.json"; BUNDLE = f"{WORK}/bundle_piezo"
os.environ.pop("WANDB_API_KEY", None); os.environ["WANDB_MODE"] = "offline"
Path(OUT).mkdir(parents=True, exist_ok=True)

## 4. Prueba de tiempo en la A100 (primero, antes de gastar)
Corre unos cientos de pasos de B, E e I en converso y directo, float64 en un proceso aparte por prueba, y escribe `timing_probe.json` (s/paso y s/recalculo reales), `projection.json` y la tabla de horas por prioridad. La tasa de unidades/hora la pone usted (celda de parametros); la ganancia de concurrencia se mide con 2 procesos simultaneos.

In [ ]:
# 5. PRUEBA DE TIEMPO + proyeccion de costo
rate = f"--units-per-hour {UNIDADES_POR_HORA}" if UNIDADES_POR_HORA else ""
!python {RUNNER} probe --manifest {MANIFEST} --bundle {BUNDLE} --out {OUT} --steps {PASOS_PRUEBA} --concurrency 2 --scratch /content/x28_probe {rate}
# Lea la tabla: horas de pared por prioridad y (si declaro la tasa) unidades de computo acumuladas. Decida MAX_PRIORIDAD (y PROCESOS_POR_GPU) con eso.

In [ ]:
# 6. LANZAR (deje la pestana abierta; si se corta, vaya a la celda 7). Crea OUT/STOP para detener el despacho de forma ordenada desde otra pestana de Drive.
extra = ""
if UNIDADES_POR_HORA and MAX_UNIDADES: extra += f" --max-compute-units {MAX_UNIDADES} --units-per-hour {UNIDADES_POR_HORA}"
!python {RUNNER} run --manifest {MANIFEST} --bundle {BUNDLE} --out {OUT} --scratch {SCRATCH} --gpus 0 --jobs-per-gpu {PROCESOS_POR_GPU} --max-priority {MAX_PRIORIDAD} {('--only-problem ' + SOLO_PROBLEMA) if SOLO_PROBLEMA else ''} --cutoff-hours {TOPE_DESPACHO_H} --hard-cap-hours {TOPE_KILL_H} --fit-check {extra}

In [ ]:
# 7. REANUDAR tras un corte (nueva sesion: repita las celdas 1-3, no hace falta repetir la 5). Salta todo lo que ya esta en OUT; los trabajos que se cortaron se repiten desde cero.
!python {RUNNER} status --manifest {MANIFEST} --out {OUT}
!python {RUNNER} run --manifest {MANIFEST} --bundle {BUNDLE} --out {OUT} --scratch {SCRATCH} --gpus 0 --jobs-per-gpu {PROCESOS_POR_GPU} --max-priority {MAX_PRIORIDAD} {('--only-problem ' + SOLO_PROBLEMA) if SOLO_PROBLEMA else ''} --cutoff-hours {TOPE_DESPACHO_H} --hard-cap-hours {TOPE_KILL_H} --fit-check

In [ ]:
# 8. EMPAQUETAR resultados a Drive (zip con JSON, npz, ledger, logs y carpetas wandb offline) y mostrar su SHA256
import time
stamp = time.strftime("%Y%m%d-%H%M")
dest = Path(OUT).parent / f"resultados_piezo_{stamp}"
shutil.make_archive(str(dest), "zip", OUT)
zf = Path(str(dest) + ".zip"); print(zf, round(zf.stat().st_size / 2**20, 1), "MB")
print("SHA256:", hashlib.sha256(zf.read_bytes()).hexdigest())
!python {RUNNER} status --manifest {MANIFEST} --out {OUT}

## 9. Despues (en su PC, no aqui)
1. Descargue `resultados_*.zip` de Drive y descomprimalo en una carpeta bajo `experiments/mechanisms/EXP-0028-failure-attribution/results/`.
2. Sincronice W&B con la clave del `.env` solo en memoria: `python wandb_tools/sync_wandb.py <carpeta>` (dry-run) y luego `--apply`.
3. **Desconecte y elimine el entorno de ejecucion** (Entorno de ejecucion > Cancelar conexion y eliminar) para dejar de consumir unidades.